# 14.01 · Local models with Ollama

**Module 14 · Local Models & Fine-Tuning** · Part 2 (Pro) · elective

| | |
|---|---|
| ⏱️ Estimated time | 2–3 h |
| 📋 Prerequisites | Modules 01–02 (models, tools) and Module 06 (RAG). Read the module README first. |
| 🔑 Keys | none: Ollama runs on your computer · optional `GROQ_API_KEY` for the Groq comparison |
| 🧠 Kernel | **Python (lc-mastery)** |
| ✅ Docs verified | 2026-10-16 |

> **How to use this notebook:** read each Markdown cell, then run the code cell below it with **Shift + Enter**, top to bottom.
> Cell markers: **🧪** offline (scripted model, no network, free) · **🌐** network, no tokens · **💸** calls a model (uses your free-tier quota).
> Without a key, 💸 cells print `⏭️ Skipped` and everything else still runs, so **Kernel → Restart Kernel and Run All Cells…** always completes.

## Learning objectives

1. **Install and check** Ollama on Windows, and choose a model that fits your RAM (and GPU, if you have one).
2. **Call** a local model from LangChain with `ChatOllama`, streamed, and measure load time and tokens per second.
3. **Verify** structured output and tool calling on the local model, instead of trusting a capability tag.
4. **Embed** text locally with `OllamaEmbeddings` and compare it with the offline hashing baseline.
5. **Compare** the same RAG answers on Groq and on your machine: quality, latency, cost and privacy.
6. **Explain** what quantization (`Q4_K_M`) and thinking mode cost you on a CPU-only laptop.

## 📚 Official reading (verified 2026-10-16)

| Page | Read it for |
|---|---|
| [Ollama quickstart](https://docs.ollama.com/quickstart) | `ollama pull`, `ollama run`, `ollama list` |
| [Ollama on Windows](https://docs.ollama.com/windows) | Installer, GPU support, where models are stored (`OLLAMA_MODELS`) |
| [Ollama tool calling](https://docs.ollama.com/capabilities/tool-calling) | Which models take tools and how |
| [ChatOllama](https://docs.langchain.com/oss/python/integrations/chat/ollama) | `bind_tools`, structured output, `reasoning`, logprobs |
| [Ollama library](https://ollama.com/library) | Model sizes and capability tags (`tools`, `vision`, `thinking`) |

> **Why this matters at work.** Orbita Cloud's legal team won't let customer tickets leave the company network, and finance asks what the Groq bill would be at 100× the volume. A local model answers both questions: the data stays on the machine and the marginal cost is electricity. The price is speed and quality on ordinary hardware, which this lesson measures instead of guessing.

> 🖥️ **CPU-only is fine for this module.** Most laptops (including ones with only an integrated Intel GPU) run Ollama on the **CPU**. A 4B model at 4-bit (`qwen3:4b`, 2.5 GB) answers in seconds to a minute; a 20B model (`gpt-oss:20b`, 14 GB) fits in 32 GB of RAM but writes only a few tokens per second. Fine-tuning (14.04) runs on a free Colab GPU, never on your laptop.

---
## Step 0 · Install and set up

In [ ]:
%pip install --quiet "langchain>=1.4.4,<2" "langchain-core>=1.6.9,<2" "langchain-ollama>=1.1.0,<2" "langchain-groq>=1.1.3,<2"

**You should now see** *"Note: you may need to restart the kernel to use updated packages."*, possibly below a few lines from pip. IPython prints that note after **every** `%pip`; restart only if a package was **upgraded** after you had already imported it (Lesson 00.05). `%pip` always installs into **this kernel's** environment; `!pip` might not (Lesson 00.02).

In [ ]:
from importlib import metadata

for package in ("langchain", "langchain-ollama", "langchain-groq",):
    print(f"{package:<22} {metadata.version(package)}")

In [ ]:
# Setup 1: Find the repository root (Jupyter starts in modules/14-local-models-and-fine-tuning/).
import importlib.util
import os
import sys
from pathlib import Path

# Optional: set LC_MASTERY_ROOT to your clone if Jupyter starts outside it (no paths are hardcoded).
_override = os.environ.get("LC_MASTERY_ROOT", "").strip()
REPO_ROOT = next(
    (p for p in ([Path(_override).expanduser()] if _override else [Path.cwd(), *Path.cwd().parents])
     if (p / "pyproject.toml").is_file() and (p / "src" / "mastery").is_dir()),
    None,
)
if REPO_ROOT is None:
    raise FileNotFoundError("Could not find the course repo. Start Jupyter inside your clone (or set LC_MASTERY_ROOT to it).")

# Setup 2: Import `mastery` from THIS clone's src/, like pytest does (pythonpath = ["src"]), and check
# that the env's own install (00.02 guide, Step 5) points here too: terminal commands use that one.
SRC_DIR = (REPO_ROOT / "src").resolve()
installed_spec = importlib.util.find_spec("mastery")
if sys.path[0] != str(SRC_DIR):
    sys.path.insert(0, str(SRC_DIR))
import mastery

print("Repo root:", REPO_ROOT)
print("mastery  :", mastery.__version__, "from", mastery.__file__)
installed_at = installed_spec.origin if installed_spec else None
if not Path(mastery.__file__).resolve().is_relative_to(SRC_DIR):
    print("⚠️ This kernel had already imported mastery from another folder: restart the kernel, run from the top.")
elif installed_at is None or not Path(installed_at).resolve().is_relative_to(SRC_DIR):
    print("⚠️ This env's own mastery install does not use this clone's src/:", installed_at or "(not installed)")
    print("   This notebook uses this clone's src/ anyway, but terminal commands (python -m mastery...) do not.")
    print('   Fix once (Git Bash, course env active, root of THIS clone): python -m pip install -e ".[dev]"')

In [ ]:
KEYS_TO_CHECK = ('GROQ_API_KEY',)
LANGSMITH_PROJECT = 'm14-ollama-local-models'
# Setup 3: Load .env, check keys WITHOUT printing them, read model IDs from config, look for a local Ollama.
import os

from mastery.config import env_status, is_set, load_env, load_settings
from mastery.local import ollama_status

dotenv_path = load_env()
settings = load_settings(load_dotenv_file=False)   # .env was just loaded.
HAVE_GROQ_KEY = is_set("GROQ_API_KEY")
HAVE_HF_TOKEN = is_set("HF_TOKEN")
HAVE_LANGSMITH_KEY = is_set("LANGSMITH_API_KEY")
os.environ.setdefault("LANGSMITH_PROJECT", LANGSMITH_PROJECT)   # m14-<lesson>: one tracing project per lesson
OLLAMA = ollama_status(timeout=3.0)                              # never raises; 3 s if nothing is listening
HAVE_OLLAMA = OLLAMA.running

print(".env      :", dotenv_path or "not found (fine if keys are set in your shell)")
for name, status in env_status(*KEYS_TO_CHECK).items():
    print(f"{name:<18}: {status}")
print("project   :", os.environ["LANGSMITH_PROJECT"])
print("fast      :", settings.model_fast, "(Groq)")
print("local     :", settings.model_local, "| embeddings:", settings.model_local_embed)
print("ollama    :", f"{OLLAMA.version} at {OLLAMA.base_url}, models: {OLLAMA.names() or 'none pulled'}" if HAVE_OLLAMA
      else f"not running at {OLLAMA.base_url}")
if not HAVE_OLLAMA:
    print("\n⚠️ No Ollama server: 🖥️ local cells will print ⏭️ Skipped. Start the Ollama app (or `ollama serve`) and re-run.")
if not HAVE_GROQ_KEY:
    print("⚠️ No GROQ_API_KEY: 💸 Groq cells will print ⏭️ Skipped; 🧪 cells still teach the concept.")

In [ ]:
# Setup 4: Silence known, harmless noise so real problems stand out.
import os
import warnings
from importlib.metadata import version

# langsmith warns once when a Client is created without a key; offline cells never send anything.
warnings.filterwarnings("ignore", message="API key must be provided when using hosted LangSmith API")
# Same as Lesson 02.04: older langchain-core 1.6.x Pydantic serializer warning about the injected runtime.
# Only older releases raise it; on current ones this filter is skipped.
if tuple(int("".join(c for c in x if c.isdigit()) or 0) for x in version("langchain-core").split(".")[:3]) < (1, 6, 11):
    warnings.filterwarnings("ignore", message="Pydantic serializer warnings", category=UserWarning)
# Preview models warn once per model build; the lesson says so in Markdown already.
from mastery.models import PreviewModelWarning
warnings.filterwarnings("once", category=PreviewModelWarning)

In [ ]:
def table(rows, columns):
    """Print dict rows as an aligned text table (one comparison cell per Part)."""
    widths = {c: max(len(c), *(len(str(r.get(c, ""))) for r in rows)) for c in columns}
    print("  ".join(c.ljust(widths[c]) for c in columns))
    for r in rows:
        print("  ".join(str(r.get(c, "")).ljust(widths[c]) for c in columns))

import time

from mastery.local import fits_in_memory, measure, require_model

LOCAL = settings.model_local          # MASTERY_MODEL_LOCAL, default qwen3:4b
EMBED = settings.model_local_embed    # MASTERY_MODEL_LOCAL_EMBED, default nomic-embed-text
RAM_GB = 32                           # ← your computer's RAM (Windows: Settings → System → About)
VRAM_GB = 0                           # ← dedicated GPU memory; 0 for integrated graphics (Intel UHD/Iris)
HAVE_LOCAL = HAVE_OLLAMA and OLLAMA.get(LOCAL) is not None
SKIP_LOCAL = (f"⏭️ Skipped: {LOCAL} is not pulled (in a terminal: ollama pull {LOCAL})" if HAVE_OLLAMA
              else "⏭️ Skipped: no Ollama server (see the setup cell).")
if HAVE_OLLAMA and not HAVE_LOCAL:
    print(SKIP_LOCAL)

---
## Part 1 · Install, pull, and choose a model that fits

In a terminal (PowerShell):

```powershell
ollama --version          # update from https://ollama.com/download if it is older than a few months
ollama pull qwen3:4b      # 2.5 GB, tools + thinking; the course default for local calls
ollama pull nomic-embed-text   # 274 MB, embeddings
ollama list
```

The Windows app starts the server in the background (tray icon); the API listens on `http://localhost:11434`, **only on your machine**. Re-run the setup cell afterwards: it lists what you pulled.

In [ ]:
# 🧪 Offline: which models fit this machine? (size on disk ≈ memory needed for the weights at 4-bit)
CANDIDATES = {"qwen3:0.6b": 0.5, "llama3.2:3b": 2.0, "qwen3:4b": 2.5, "qwen3.5:4b": 3.4, "qwen3:8b": 5.2,
              "gpt-oss:20b": 14.0, "qwen3:30b": 19.0, "llama3.3:70b": 43.0}
table([{"model": m, "GB": gb, "runs on": fits_in_memory(gb, RAM_GB, vram_gb=VRAM_GB)} for m, gb in CANDIDATES.items()],
      ["model", "GB", "runs on"])

**You should now see** with `RAM_GB = 32, VRAM_GB = 0`: everything up to `qwen3:30b` on `cpu`, `llama3.3:70b` → `no`. With 16 GB, `gpt-oss:20b` drops to `no`.

### What just happened?

`fits_in_memory` adds about 20% for the context (KV cache) and 2 GB for Windows. "Runs" isn't "runs well": on a CPU, speed falls roughly in proportion to model size, so a 20B model is about 5× slower than a 4B one. The `Q4_K_M` in a model's details means 4-bit weights: about a quarter of the 16-bit size, with a small quality loss.

In [ ]:
# 🖥️ (HAVE_OLLAMA) What the server has: size, quantization, context window, capabilities.
if HAVE_OLLAMA:
    table([{"model": m.name, "GB": m.size_gb, "params": m.parameters, "quant": m.quantization,
            "context": m.context_length, "capabilities": ", ".join(m.capabilities)} for m in OLLAMA.models],
          ["model", "GB", "params", "quant", "context", "capabilities"])
else:
    print("⏭️ Skipped: no Ollama server (see the setup cell).")

**You should now see** your pulled models, for example `qwen3:4b  2.5  4.0B  Q4_K_M  40960  completion, tools, thinking`, or `⏭️ Skipped`.

---
## Part 2 · The first local call, streamed and timed

> 🖥️ Runs **on your computer** through Ollama: no key, no cost, nothing leaves the machine. On a **CPU-only** laptop (no NVIDIA GPU) expect 10–60 s for the first call (the model loads into RAM) and roughly 5–15 tokens per second for a 4B model afterwards; the course gives local calls a 300 s timeout. Without a running Ollama it prints `⏭️ Skipped`.

In [ ]:
# 🖥️ (HAVE_OLLAMA) get_model(provider="ollama"): ChatOllama with a 300 s timeout and thinking off.
from mastery.models import get_model

if HAVE_OLLAMA and HAVE_LOCAL:
    require_model(LOCAL, OLLAMA)                     # clear message + the exact `ollama pull` if missing
    llm = get_model("fast", "ollama", temperature=0)
    first = measure(llm, "In two sentences: what does a customer support team use a ticket queue for?")
    again = measure(llm, "In two sentences: why do support teams tag tickets by product?")
    table([{"call": "first (loads the model)", **first.__dict__}, {"call": "second (model in RAM)", **again.__dict__}],
          ["call", "seconds", "load_s", "output_tokens", "tokens_per_s"])
    print("\n", again.text)
else:
    print(SKIP_LOCAL)

**You should now see** two rows: the first with a `load_s` of several seconds (the weights are read into RAM), the second without it, and the tokens per second your CPU manages (often 5–15 for a 4B model on a laptop CPU). Then the answer.

### What just happened?

`get_model("fast", "ollama")` built a `ChatOllama` for `MASTERY_MODEL_LOCAL`. The course adds three things for local use: **thinking off** (`reasoning=False`; Qwen3 otherwise writes a long hidden reasoning first, minutes on a CPU), a **300 s timeout** instead of 60 s, and `num_predict` when you pass `max_tokens`. Ollama keeps a model loaded for 5 minutes after the last call (`keep_alive`), which is why the second call is faster.

In [ ]:
# 🖥️ (HAVE_OLLAMA) Streaming: time to first token is what a user waits for.
if HAVE_OLLAMA and HAVE_LOCAL:
    start, first_token, parts = time.perf_counter(), None, []
    for chunk in llm.stream("List three things a status page should show during an outage."):
        if chunk.content and first_token is None:
            first_token = time.perf_counter() - start
        parts.append(chunk.content)
    print(f"first token after {first_token:.2f} s, all after {time.perf_counter() - start:.2f} s\n")
    print("".join(parts))
else:
    print(SKIP_LOCAL)

**You should now see** the first token after about a second (the model is loaded) and the full answer some seconds later.

---
## Part 3 · Don't trust the tag: test structured output and tools

In [ ]:
# 🖥️ (HAVE_OLLAMA) The same two checks the course runs on Groq models, on the local one.
from typing import Literal

from langchain_core.tools import tool
from pydantic import BaseModel, Field


class Triage(BaseModel):
    queue: Literal["billing", "outage", "account_access", "bug", "how_to", "feature_request"]
    urgent: bool = Field(description="True if customers are blocked right now")


@tool
def latest_invoice(customer_id: str) -> str:
    """Return the latest invoice for a customer ID such as C-1002."""
    return f"{customer_id}: INV-2026-0912, 1,240 USD, due 2026-11-05"


TICKETS = ["Dashboard is down for our whole team since 9am, we can't see any readings!",
           "Could you add dark mode to the mobile app?",
           "I was charged twice for October, please refund one payment."]
if HAVE_OLLAMA and HAVE_LOCAL:
    triage = llm.with_structured_output(Triage)
    for text in TICKETS:
        start = time.perf_counter()
        result = triage.invoke(f"Triage this support ticket: {text}")
        print(f"{time.perf_counter() - start:5.1f} s", result, "|", text[:45])
    reply = llm.bind_tools([latest_invoice]).invoke("What is the latest invoice for customer C-1002?")
    print("tool calls:", reply.tool_calls or "none: the model answered in text")
else:
    print(SKIP_LOCAL)

**You should now see** three `queue=... urgent=...` lines (outage/True, feature_request/False, billing/False if the model is good) and one `latest_invoice` tool call with `customer_id: 'C-1002'`.

### What just happened?

`with_structured_output` on `ChatOllama` sends a JSON schema that Ollama enforces while decoding, so the reply always parses. Whether the **label is right** is a quality question: small local models misroute more tickets than `gpt-oss-20b`. Lesson 14.03 measures that on 200 tickets instead of three. Tool calling depends on the model's chat template: a model without the `tools` capability makes Ollama return an error ("does not support tools"), one with it can still pick the wrong tool, so always test with your own tools.

---
## Part 4 · Local embeddings

In [ ]:
# 🖥️ (HAVE_OLLAMA) Meaning vs shared words: OllamaEmbeddings against the offline hashing baseline.
from mastery.memory import HashingEmbeddings

QUERY = "my weather station stopped sending data"
DOCS = ["The bridge shows a red light after 15 minutes without a report.",
        "Invoices are emailed on the first business day of the month.",
        "Readings stop when the station battery is too cold to charge."]


def ranked(embeddings):
    q = embeddings.embed_query(QUERY)
    scores = [sum(a * b for a, b in zip(q, d)) / ((sum(a * a for a in q) * sum(b * b for b in d)) ** 0.5 or 1)
              for d in embeddings.embed_documents(DOCS)]
    return sorted(zip(scores, DOCS), reverse=True)


print("hashing (offline, words only):")
for score, doc in ranked(HashingEmbeddings()):
    print(f"  {score:.2f}  {doc}")
if HAVE_OLLAMA and OLLAMA.get(EMBED):
    from langchain_ollama import OllamaEmbeddings
    print(f"{EMBED} (local, meaning):")
    for score, doc in ranked(OllamaEmbeddings(model=EMBED, base_url=settings.ollama_base_url)):
        print(f"  {score:.2f}  {doc}")
elif HAVE_OLLAMA:
    print(f"⏭️ Skipped: {EMBED} is not pulled (ollama pull {EMBED}).")
else:
    print("⏭️ Skipped: no Ollama server (see the setup cell).")

**You should now see** the hashing baseline scoring near zero for most sentences (no shared words besides "station"), and the local embedding model ranking the two outage-like sentences above the invoice one.

### What just happened?

A local embedding model gives meaning-based retrieval with zero data leaving the machine, and it's fast even on a CPU (it's small). Indexing a whole corpus locally is the first half of Project 14: the Project 06 assistant with local embeddings **and** a local chat model.

---
## Part 5 · Groq vs your machine on the same RAG questions

Six questions from the Project 06 test set. The context comes from the same BM25 search for both models, so only the reader changes.

In [ ]:
# 🖥️💸 (HAVE_OLLAMA / HAVE_GROQ_KEY) Same prompt, two readers: right answer?, seconds, tokens per second.
import json

from mastery.rag import format_context, load_corpus, split_documents
from mastery.rag.retrieve import BM25

rows = [json.loads(line) for line in (REPO_ROOT / "data" / "samples" / "rag" / "project06-test-set.jsonl").read_text(encoding="utf-8").splitlines()]
QUESTIONS = [r for r in rows if r["id"] in ("t01", "t11", "t14", "t28", "t29", "t38")]
index = BM25(split_documents(load_corpus(REPO_ROOT / "data" / "samples" / "rag" / "corpus"), preset="markdown", chunk_size=800, chunk_overlap=100))
PROMPT = ("Answer the question in one short sentence using only the sources. If they don't answer it, say you don't know.\n\n"
          "{context}\n\nQuestion: {question}")

readers = {}
if HAVE_LOCAL:
    readers[f"ollama:{LOCAL}"] = llm
if HAVE_GROQ_KEY:
    readers[f"groq:{settings.model_fast}"] = get_model("fast", temperature=0)
results = []
for name, reader in readers.items():
    for q in QUESTIONS:
        s = measure(reader, PROMPT.format(context=format_context(index.search(q["question"], k=4)), question=q["question"]))
        results.append({"reader": name, "id": q["id"], "right": q["answer_contains"].casefold() in s.text.casefold(),
                        "seconds": s.seconds, "tokens_per_s": s.tokens_per_s})
if results:
    table(results, ["reader", "id", "right", "seconds", "tokens_per_s"])
    print()
    summary = [{"reader": n, "right": f"{sum(r['right'] for r in results if r['reader'] == n)}/{len(QUESTIONS)}",
                "mean s": round(sum(r["seconds"] for r in results if r["reader"] == n) / len(QUESTIONS), 2)} for n in readers]
    table(summary, ["reader", "right", "mean s"])
else:
    print("⏭️ Skipped: needs a running Ollama with", LOCAL, "and/or GROQ_API_KEY (see the setup cell).")

> 💸 With `GROQ_API_KEY`: 6 calls to `gpt-oss-20b`, about 4K tokens. Local calls are free.

**You should now see** one row per question and reader, then a summary such as `ollama:qwen3:4b 5/6 9.8 s` and `groq:openai/gpt-oss-20b 6/6 0.6 s` (your numbers will differ; a CPU is 10–30× slower than Groq here).

### What just happened?

Six questions is a smoke test, not an evaluation: the 95% confidence interval on 5/6 runs from about 44% to 97%. Project 14 runs the whole test set with confidence intervals and a paired comparison. Even so, the shape is typical: on grounded questions a 4B local model is often close to a hosted 20B one in **quality**, and far behind in **latency** on a CPU.

In [ ]:
# 🖥️ (HAVE_OLLAMA, RUN_STRETCH) Thinking mode on a CPU: same question, reasoning off vs on.
RUN_STRETCH = False   # set True to try it (can take minutes on a CPU)
if HAVE_OLLAMA and HAVE_LOCAL and RUN_STRETCH:
    for think in (False, True):
        thinker = get_model("fast", "ollama", temperature=0, reasoning=think)
        s = measure(thinker, "A customer was charged 1,240 USD instead of 124 USD. By what factor? Answer with the number.")
        print(f"reasoning={think!s:<5} {s.seconds:6.1f} s, {s.output_tokens:4} tokens → {s.text[:60]!r}")
elif HAVE_LOCAL:
    print("⏭️ Skipped (optional): set RUN_STRETCH = True to compare thinking off and on.")
else:
    print(SKIP_LOCAL)

**You should now see** `⏭️ Skipped (optional)` by default; with `RUN_STRETCH = True`, thinking on uses several times more tokens and seconds for the same answer (`10`).

---
## ⚠️ Pitfalls and troubleshooting

| Symptom | Cause | Fix |
|---|---|---|
| `ConnectError` / `not running at http://localhost:11434` | Ollama app not started | Start it from the Start menu (or `ollama serve`), re-run the setup cell |
| `model "qwen3:4b" not found` | Not pulled, or a different tag | `ollama pull qwen3:4b`; tags must match exactly (`qwen3:4b` ≠ `qwen3`) |
| First call takes a minute, then fast | Model loading into RAM | Normal; keep it loaded (`keep_alive`) or warm it up at startup |
| Answer starts with `<think>` or takes minutes | Thinking mode on | `reasoning=False` (the course default for local calls) |
| `ReadTimeout` on a CPU | Long answer on a slow machine | `max_tokens`, a smaller model, or a longer `timeout` |
| Windows says the disk is full | Models are GBs in `%HOMEPATH%\.ollama` | `ollama rm <model>`, or move them with `OLLAMA_MODELS` |
| New model tag "not supported" | Old Ollama version | Update Ollama (new model families need new releases) |

## ⚠️ Production notes

> ⚠️ **Production note: local is private only if it stays local.** (Part 1)
> Ollama listens on `localhost` by default; don't set `OLLAMA_HOST=0.0.0.0` on a laptop or expose port 11434 to a network without authentication in front of it. Check each model's **license** (Llama, Gemma and Qwen differ) before using it commercially.

> ⚠️ **Production note: one laptop is not a server.** (Parts 2, 5)
> A CPU serves one request at a time at a few tokens per second. For a team, run Ollama (or vLLM, Lesson 14.04) on a GPU server, set `OLLAMA_NUM_PARALLEL`, measure under load (Lesson 12.06's load test works against it), and keep a hosted model as fallback.

## 🏋️ Exercises

1. **(Easy)** Pull `qwen3:0.6b` and run Part 2 with `MASTERY_MODEL_LOCAL=qwen3:0.6b`. How much faster is it, and what goes wrong in the answers?
2. **(Medium)** Run Part 3 with 10 tickets of your own. How many queues does the local model get right? Write them down for Lesson 14.03.
3. **(Medium)** Make Part 5 fair on output length: set `max_tokens=60` for both readers and compare seconds again.
4. **(Stretch)** If you have 32 GB of RAM, pull `gpt-oss:20b` (14 GB) and add it to Part 5. Is it worth the wait on your CPU?

<details>
<summary><b>Show solutions</b></summary>

**1.** In PowerShell `ollama pull qwen3:0.6b`, then in the notebook `llm = get_model(model="ollama:qwen3:0.6b", temperature=0)` and re-run Part 2. Expect 3–5× more tokens per second and confident wrong facts (a 0.6B model will happily invent what Ollama is). Small models need grounding (RAG) even more than big ones.

**2.** `results = [(t, triage.invoke(f"Triage this support ticket: {t}").queue) for t in my_tickets]`, then count matches with your labels. Keep the list; 14.03 builds a 900-ticket version.

**3.** `get_model("fast", "ollama", temperature=0, max_tokens=60)` and `get_model("fast", temperature=0, max_tokens=60)`; seconds now compare speed, not verbosity.

**4.** `readers["ollama:gpt-oss:20b"] = get_model(model="ollama:gpt-oss:20b", temperature=0)`. On a CPU expect 1–4 tokens per second: fine for overnight batch jobs, not for a chat window.
</details>

## 🔎 Docs drill (5 min)

Where does Ollama on Windows store downloaded models, and which environment variable moves them?

Try it before you open the answer: use the official docs, the API reference, or the installed source (`??` in Jupyter).

<details>
<summary>Path to the answer</summary>

1. Open [Ollama on Windows](https://docs.ollama.com/windows) and find *Changing Model Location*.
2. Models live under `%HOMEPATH%\.ollama` by default.
3. Set the user environment variable **`OLLAMA_MODELS`** to another folder, then quit and restart the Ollama app.

</details>

## ✅ Checkpoint

- [ ] Ollama runs and the setup cell lists my models.
- [ ] I know which models fit my RAM, and my CPU's tokens per second for a 4B model.
- [ ] I tested structured output and tool calling on my local model.
- [ ] I compared local and Groq answers on the same six questions.
- [ ] Restart & Run All passes with Ollama stopped (cells skip); outputs cleared before saving.

## 🔎 Research checkpoint

On the [Ollama library](https://ollama.com/library), filter by **tools** and log in [`docs/research-log.md`](../../docs/research-log.md): the best model that fits your RAM (and GPU), its size and context window, and your measured tokens per second.

## Summary

- Ollama serves local models on `localhost:11434`; `get_model("fast", "ollama")` gives a `ChatOllama` with thinking off and a 300 s timeout.
- Pick by memory: a 4-bit model needs about its file size plus 20% for context; on a CPU, speed drops with size.
- Capability tags are a start; test structured output and tools with your own schema and tools.
- Local embeddings are small and fast even on a CPU: the easiest local win.
- On grounded questions a local 4B model can be close in quality and far behind in speed; measure before deciding.

---
## ➡️ What's next

**You're done with this lesson when** every box in the ✅ Checkpoint is ticked and **Kernel → Restart Kernel and Run All Cells…** finishes without errors.

1. Clear outputs before you save (**Edit → Clear Outputs of All Cells**), then **File → Save** (`Ctrl + S`).
2. Optional: log your research checkpoint in [`docs/research-log.md`](../../docs/research-log.md).
3. Next: **Lesson 14-02-hugging-face.ipynb** → [`14-02-hugging-face.ipynb`](14-02-hugging-face.ipynb).

[Module 14 README](README.md) · [Course outline](../../COURSE_OUTLINE.md)